<a href="https://colab.research.google.com/github/Aria05MC/PlayGround/blob/main/AIAgentOpenCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **All setup might needed**

In [1]:
#@title Setup python library
!pip install psutil gputil

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for gputil: filename=gputil-1.4.0-py3-none-any.whl size=7434 sha256=8d2dc617343d3ca29ab2739bce312a41d76135f8b88b0254923bea54621027de
  Stored in directory: /root/.cache/pip/wheels/ec/ae/c6/43c88094e51d00abb1f20a8acd65dc7af7dce93057c46f2eff
Successfully built gputil


In [2]:
#@title Print spesifikasi mesin
import os
import platform
import sys
import psutil

try:
    import GPUtil
except ImportError:
    GPUtil = None


def get_gpu_info():
    """Mengambil informasi GPU dan VRAM (MB)."""
    gpus_data = []
    if GPUtil:
        try:
            gpus = GPUtil.getGPUs()
            for gpu in gpus:
                gpus_data.append({
                    "name": gpu.name,
                    "vram_total_mb": gpu.memoryTotal,
                    "vram_free_mb": gpu.memoryFree,
                    "vram_total_gb": round(gpu.memoryTotal / 1024, 2),
                    "vram_free_gb": round(gpu.memoryFree / 1024, 2),
                })
        except Exception as e:
            print(f"[Warning] Gagal mengambil info GPU via GPUtil: {e}")
    return gpus_data


def get_system_specs():
    """Mengambil spesifikasi detail perangkat keras sistem."""
    # Memory
    ram = psutil.virtual_memory()
    ram_total_gb = round(ram.total / (1024**3), 2)
    ram_available_gb = round(ram.available / (1024**3), 2)

    # CPU
    cpu_name = platform.processor() or "Unknown CPU"
    cpu_cores_physical = psutil.cpu_count(logical=False)
    cpu_cores_logical = psutil.cpu_count(logical=True)

    # Storage
    disk = psutil.disk_usage("/")
    disk_free_gb = round(disk.free / (1024**3), 2)

    # GPU
    gpus = get_gpu_info()

    return {
        "os": f"{platform.system()} {platform.release()} ({platform.machine()})",
        "cpu_name": cpu_name,
        "cpu_physical_cores": cpu_cores_physical,
        "cpu_logical_cores": cpu_cores_logical,
        "ram_total_gb": ram_total_gb,
        "ram_available_gb": ram_available_gb,
        "disk_free_gb": disk_free_gb,
        "gpus": gpus,
    }


def analyze_ai_capabilities(specs):
    """Menganalisis kemampuan sistem berdasarkan VRAM dan RAM untuk menjalankan LLM/Diffusers."""
    gpus = specs["gpus"]
    total_vram_gb = sum(g["vram_total_gb"] for g in gpus) if gpus else 0
    ram_gb = specs["ram_total_gb"]

    # Menentukan tier performa utama (GPU VRAM diutamakan, fallback ke System RAM via GGUF/llama.cpp)
    effective_mem = max(total_vram_gb, ram_gb * 0.6)  # CPU offloading penalty

    recommendations = []

    # 1. Analisis Model Bahasa (LLM)
    if total_vram_gb >= 24:
        llm_tier = "Ultra / High-End"
        llm_models = [
            "**Llama 3 / 3.1 70B** (Quantization Q4_K_M / EXL2)",
            "**Mixtral 8x7B / 8x22B**",
            "**Command R+** (Q4)",
            "**Qwen 2.5 32B / 72B** (Q4_K_M)",
            "Semua model 7B - 14B pada FP16/Uncensored/Full Precision",
        ]
        llm_frameworks = [
            "vLLM / TGI (Inference Sangat Cepat)",
            "Ollama / LM Studio",
            "Text Generation WebUI (oobabooga)",
        ]
    elif total_vram_gb >= 12 or ram_gb >= 32:
        llm_tier = "Mid-to-High End"
        llm_models = [
            "**Llama 3 / 3.1 8B** (FP16 / Uncensored)",
            "**Qwen 2.5 14B / 32B** (Q4_K_M / Q5_K_M)",
            "**Mistral 7B / Nemo 12B** (FP16 / Q8)",
            "**Gemma 2 9B / 27B** (Q4)",
            "**Goliath / Llama 70B** (Q2_K / Q3 via CPU+GPU Offloading)",
        ]
        llm_frameworks = [
            "Ollama / LM Studio",
            "llama.cpp / KoboldCPP",
            "Text Generation WebUI",
        ]
    elif total_vram_gb >= 6 or ram_gb >= 16:
        llm_tier = "Mid Range"
        llm_models = [
            "**Llama 3 8B** (Quantized Q4_K_M / Q5_K_M)",
            "**Qwen 2.5 7B / 14B** (Quantized Q4_K_M)",
            "**Phi-3 / Phi-3.5 Mini (3.8B)** (FP16 atau Q8)",
            "**Mistral 7B Instruct** (Q4_K_M)",
        ]
        llm_frameworks = [
            "Ollama",
            "LM Studio",
            "llama.cpp (GPU offload sebagian layer)",
        ]
    else:
        llm_tier = "Entry Level / CPU-Only"
        llm_models = [
            "**Phi-3 Mini (3.8B)** (Q4_K_M)",
            "**Qwen 2.5 1.5B / 3B** (Q4 / Q5)",
            "**Llama 3.2 1B / 3B**",
            "**TinyLlama 1.1B**",
        ]
        llm_frameworks = [
            "Ollama (Mode CPU)",
            "llama.cpp (Menggunakan Thread CPU)",
        ]

    # 2. Analisis Generator Gambar (Image Gen)
    if total_vram_gb >= 16:
        img_tier = "Full Speed / Unrestricted"
        img_models = [
            "**Flux.1 (Dev / Schnell)** pada FP16 / FP8",
            "**Stable Diffusion XL (SDXL 1.0)** - Native Resolution 1024x1024",
            "**Pony Diffusion V6 XL** & Custom LoRA",
            "**Stable Diffusion 1.5 / 2.1** (High-batch generation)",
        ]
    elif total_vram_gb >= 8:
        img_tier = "Good Performance"
        img_models = [
            "**Flux.1 Schnell / Dev** (NF4 / GGUF Quantized)",
            "**SDXL 1.0 / Pony Diffusion XL** (Dengan `--medvram` atau FP8)",
            "**Stable Diffusion 1.5** (Dengan banyak LoRA & ControlNet)",
        ]
    elif total_vram_gb >= 4:
        img_tier = "Basic / Low VRAM"
        img_models = [
            "**Stable Diffusion 1.5** (Dengan opsi `--lowvram` atau `--medvram`)",
            "**SDXL Turbo / LCM Models** (Resolusi rendah 512x512)",
        ]
    else:
        img_tier = "Sangat Terbatas (CPU Generation - Lambat)"
        img_models = [
            "**Stable Diffusion 1.5** via CPU (Membutuhkan waktu 2-5+ menit per gambar)",
        ]

    return {
        "llm_tier": llm_tier,
        "llm_models": llm_models,
        "llm_frameworks": llm_frameworks,
        "img_tier": img_tier,
        "img_models": img_models,
    }


def generate_markdown_report(specs, analysis, output_filename="ai_hardware_report.md"):
    """Membuat dan menyimpan laporan ke dalam file .md."""

    gpus_section = ""
    if specs["gpus"]:
        for idx, gpu in enumerate(specs["gpus"], start=1):
            gpus_section += f"- **GPU {idx}:** {gpu['name']}\n"
            gpus_section += f"  - Total VRAM: `{gpu['vram_total_gb']} GB` ({gpu['vram_total_mb']} MB)\n"
            gpus_section += f"  - Free VRAM: `{gpu['vram_free_gb']} GB` ({gpu['vram_free_mb']} MB)\n"
    else:
        gpus_section = "- **GPU:** Tidak terdeteksi GPU Nvidia/Dedicated atau GPUtil tidak terinstall.\n"

    llm_models_list = "\n".join([f"  - {m}" for m in analysis["llm_models"]])
    llm_frameworks_list = "\n".join(
        [f"  - {f}" for f in analysis["llm_frameworks"]]
    )
    img_models_list = "\n".join([f"  - {m}" for m in analysis["img_models"]])

    md_content = f"""# 📊 Laporan Analisis Hardware & Rekomendasi Model AI Lokal

Dokumen ini dihasilkan secara otomatis untuk menganalisis kelayakan sistem hardware dalam menjalankan model AI lokal (*Large Language Models* dan *Image Generators*).

---

## 🛠️ Spesifikasi Sistem Terdeteksi

- **Sistem Operasi:** `{specs['os']}`
- **CPU:** `{specs['cpu_name']}`
  - Physical Cores: `{specs['cpu_physical_cores']}`
  - Logical Threads: `{specs['cpu_logical_cores']}`
- **System RAM:**
  - Total RAM: `{specs['ram_total_gb']} GB`
  - Available RAM: `{specs['ram_available_gb']} GB`
- **Penyimpanan Bebas (Root/C:):** `{specs['disk_free_gb']} GB`
### Detail Kartu Grafis (GPU)
{gpus_section}

---

## 🤖 Rekomendasi Model AI

### 1. Large Language Models (LLM / AI Teks)
- **Kategori Performa Sistem:** `{analysis['llm_tier']}`
- **Model yang Direkomendasikan:**
{llm_models_list}
- **Aplikasi / Framework Terbaik:**
{llm_frameworks_list}

---

### 2. Image Generators (AI Gambar)
- **Kategori Performa GPU:** `{analysis['img_tier']}`
- **Model & Arsitektur yang Direkomendasikan:**
{img_models_list}
- **Software Rujukan:**
  - **ComfyUI** (Paling efisien memori & fleksibel)
  - **AUTOMATIC1111 / SD-WebUI-Forge** (User-friendly)

---

## 💡 Petunjuk & Tips Performa
1. **Model Quantization (GGUF / EXL2):** Jika VRAM terbatas, selalu gunakan format model terkompresi seperti `Q4_K_M` atau `Q5_K_M` untuk menghemat konsumsi RAM/VRAM hingga 50% tanpa penurunan kualitas yang signifikan.
2. **Offloading Layer:** Pada framework seperti *llama.cpp* atau *Ollama*, Anda bisa membagi pemrosesan layer model antara GPU dan CPU RAM jika ukuran model melebihi VRAM GPU.
3. **Penyimpanan:** Pastikan model disimpan pada **NVMe SSD** untuk waktu *loading* model yang cepat.

---
*Laporan dibuat otomatis oleh Python AI Hardware Analyzer.*
"""

    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(md_content)

    print(f"\n✅ Laporan berhasil dibuat dan disimpan ke: {os.path.abspath(output_filename)}")


def main():
    print("🔍 Menganalisis spesifikasi hardware sistem Anda...")
    specs = get_system_specs()

    print("🧠 Menghitung rekomendasi model AI lokal...")
    analysis = analyze_ai_capabilities(specs)

    output_file = "ai_hardware_report.md"
    generate_markdown_report(specs, analysis, output_file)


if __name__ == "__main__":
    main()

🔍 Menganalisis spesifikasi hardware sistem Anda...
🧠 Menghitung rekomendasi model AI lokal...

✅ Laporan berhasil dibuat dan disimpan ke: /content/ai_hardware_report.md


In [3]:
#@title Setup NodeJS
# Download and install nvm:
!curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.40.8/install.sh | bash

# in lieu of restarting the shell
!\. "$HOME/.nvm/nvm.sh"

# Download and install Node.js:
!nvm install 26

# Verify the Node.js version:
!node -v # Should print "v26.13.0".

# Verify npm version:
!npm -v # Should print "11.20.0".

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 17883  100 17883    0     0   342k      0 --:--:-- --:--:-- --:--:--  349k
=> Downloading nvm from git to '/root/.nvm'
=> Cloning into '/root/.nvm'...
remote: Enumerating objects: 545, done.
remote: Counting objects: 100% (545/545), done.
remote: Compressing objects: 100% (469/469), done.
remote: Total 545 (delta 67), reused 315 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (545/545), 600.52 KiB | 2.86 MiB/s, done.
Resolving deltas: 100% (67/67), done.
* (HEAD detached at FETCH_HEAD)
  master
=> Compressing and cleaning up git repository

=> Appending nvm source string to /root/.bashrc
=> Appending bash_completion source string to /root/.bashrc
=> You currently have modules installed globally with `npm`. These will no
=> longer be linked to the active version of Node when you install a new node
=> with `nvm`; and 

In [4]:
#@title Setup OpenCode
!npm install -g @opencode/cli

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 5 packages in 22s
⠇